# 🚀 DataMate: 1-Click Evaluation & Runner on Google Colab Pro

This notebook provides a seamless, 1-click execution of the **DataMate** agent pipeline using **vLLM** on high-performance Google Colab Pro GPUs (**NVIDIA A100 SXM 40GB/80GB** or **L4 24GB**).

### 💡 Key Benefits of Running on Colab Pro A100:
- **Single-GPU Peak Throughput (`tp=1`):** Eliminates inter-GPU PCIe tensor parallel latency, running at ~80–120 tokens/sec.
- **10x Faster Execution:** 500 questions benchmark completes in **~20–25 minutes** (compared to 3.5 hours on local/dual A30s).
- **Generous High-RAM & VRAM:** Ample space for `Qwen2.5-Coder-14B-Instruct-GPTQ-Int8` + 32k KV Cache.
- **Benchmark Modes:**
  1. `train_unseen_500`: 500 questions completely UNSEEN holdout validation (0% overlap with previous test set, stratified across 30 businesses).
  2. `train_500`: 500 questions stratified benchmark (for tracking score progress vs Run 9: +63.20%).
  3. `public_test`: Full 1,000 official test questions for final competition submission.
  4. `train_200`: 200 questions fast benchmark (~8–12 mins).
- **Automatic Result Evaluation:** Instant calculation of Penalized Accuracy (PA), breakdown by question type, and zero-false-error diagnostic CSVs.

---
> **⚠️ Important Runtime Setup:**
> In the menu above, go to **Runtime → Change runtime type**:
> - **Hardware accelerator:** `A100 GPU` (or `L4 GPU`)
> - **Runtime shape:** `High-RAM`
> Click **Save**, then run the cells in order.


## 1. Hardware & GPU Acceleration Verification

In [ ]:
# Check NVIDIA GPU allocation
!nvidia-smi

import torch
print("=" * 60)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Model    : {device_name}")
    print(f"Total VRAM   : {total_vram:.2f} GB")
    print(f"Compute Cap  : {torch.cuda.get_device_capability(0)}")
    if total_vram >= 22:
        print("✅ Excellent! Sufficient VRAM for Qwen2.5-Coder-14B with full KV cache.")
    else:
        print("⚠️ Warning: Less than 22 GB VRAM detected. L4 or A100 is recommended.")
print("=" * 60)

## 2. Mount Google Drive (Recommended for Persistence)
Mounting Google Drive allows caching the model weights and persisting submission files across Colab restarts.

In [ ]:
from google.colab import drive
import os

drive_path = "/content/drive/MyDrive/datamate"
try:
    drive.mount('/content/drive')
    os.makedirs(drive_path, exist_ok=True)
    print(f"✅ Google Drive connected! Persistence folder: {drive_path}")
except Exception as e:
    print(f"ℹ️ Drive mount skipped: {e}. Outputs will be stored in /content/result.")

## 3. Clone Repository & Setup Environment

In [ ]:
import os

REPO_URL = "https://github.com/vohoangloc-32/datamate.git"
BRANCH = "loc"
WORKDIR = "/content/datamate"

if not os.path.exists(WORKDIR):
    print(f"Cloning {REPO_URL} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL} {WORKDIR}
else:
    print(f"Pulling latest updates in {WORKDIR}...")
    !cd {WORKDIR} && git fetch origin && git checkout {BRANCH} && git reset --hard origin/{BRANCH}

%cd {WORKDIR}
!git log -1 --oneline


## 4. Install Dependencies & Run Verification Tests

In [ ]:
# Install vLLM and DataMate requirements
!pip uninstall -y torchaudio 2>/dev/null || true
!pip install -q vllm openpyxl xlrd tabulate pytest huggingface_hub pandas hf-transfer accelerate

# Run full unit test suite to verify pipeline integrity
!pytest tests/ -q


## 5. Download Model Weights to Local NVMe
Downloads `Qwen/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8` (high-speed parallel download via `huggingface_hub`).

In [ ]:
from huggingface_hub import snapshot_download
import os
import shutil

MODEL_REPO = "Qwen/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8"
LOCAL_MODEL_DIR = "/content/model_cache/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8"
DRIVE_MODEL_BACKUP = "/content/drive/MyDrive/datamate/models/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8"

if os.path.exists(DRIVE_MODEL_BACKUP) and len(os.listdir(DRIVE_MODEL_BACKUP)) > 5:
    print(f"Found model in Google Drive backup. Copying to local NVMe SSD...")
    os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)
    !cp -r {DRIVE_MODEL_BACKUP}/* {LOCAL_MODEL_DIR}/
else:
    print(f"Downloading {MODEL_REPO} from Hugging Face Hub (fast parallel download)... ")
    snapshot_download(
        repo_id=MODEL_REPO,
        local_dir=LOCAL_MODEL_DIR,
        local_dir_use_symlinks=False,
        resume_download=True,
        max_workers=8,
    )
    # Optional backup to Google Drive for instantaneous setup on future runs
    if os.path.exists("/content/drive/MyDrive/datamate"):
        print("Backing up model to Google Drive for future instant loads...")
        os.makedirs(DRIVE_MODEL_BACKUP, exist_ok=True)
        !cp -rn {LOCAL_MODEL_DIR}/* {DRIVE_MODEL_BACKUP}/

print(f"✅ Model ready at: {LOCAL_MODEL_DIR}")
!ls -lh {LOCAL_MODEL_DIR}

## 6. Run DataMate Prediction Pipeline
Executes the full pipeline with calibrated SOTA ConfidenceGate, AST Granularity Refiner, and Phase 9 Architecture.
- **Dataset:** `data/train` or `data/public_test`
- **Benchmark Options:** `train_unseen_500` (500 unseen holdout), `train_500` (500 stratified benchmark), `public_test` (1,000 official submission), `train_200`
- **Confidence Threshold:** `0.45`


In [ ]:
#@title ⚙️ Pipeline Configuration & Execution Mode { run: "auto" }
import os
import time

# Environment optimizations for single A100 / L4 GPU
os.environ["VLLM_TP"] = "1"           # Single GPU = 0 inter-GPU overhead
os.environ["VLLM_DTYPE"] = "auto"     # Native GPTQ activation dtype
os.environ["VLLM_MAX_MODEL_LEN"] = "16384"
os.environ["VLLM_MAX_NEW_TOKENS"] = "1024"

# ==============================================================================
# PIPELINE CONFIGURATION: Select Execution Mode
# ==============================================================================
#@markdown ### Choose execution mode:
MODE = "train_unseen_500" #@param ["train_unseen_500", "train_500", "public_test", "train_200"]
CONFIDENCE_THRESHOLD = "0.45" #@param ["0.45", "0.40", "0.50", "0.35"]

if MODE == "public_test":
    DATA_DIR = "/content/datamate/data/public_test"
    QUESTIONS_FILE = "/content/datamate/data/public_test/questions.jsonl"
elif MODE == "train_unseen_500":
    DATA_DIR = "/content/datamate/data/train"
    QUESTIONS_FILE = "/content/datamate/data/sample_questions_unseen_500.jsonl"
elif MODE == "train_500":
    DATA_DIR = "/content/datamate/data/train"
    QUESTIONS_FILE = "/content/datamate/data/sample_questions_500.jsonl"
elif MODE == "train_200":
    DATA_DIR = "/content/datamate/data/train"
    QUESTIONS_FILE = "/content/datamate/data/sample_questions_200.jsonl"
else:
    raise ValueError(f"Unknown MODE: {MODE}")

OUTPUT_DIR = "/content/result"

print("=" * 60)
print(f"Starting DataMate Pipeline...")
print(f"Execution Mode : {MODE}")
print(f"Data Directory : {DATA_DIR}")
print(f"Questions File : {QUESTIONS_FILE}")
print(f"Output Dir     : {OUTPUT_DIR}")
print(f"Model Dir      : {LOCAL_MODEL_DIR}")
print(f"Confidence Thresh: {CONFIDENCE_THRESHOLD}")
print("=" * 60)

t_start = time.time()
!python src/predict.py \
    --data-dir {DATA_DIR} \
    --questions-file {QUESTIONS_FILE} \
    --output-dir {OUTPUT_DIR} \
    --model-dir {LOCAL_MODEL_DIR} \
    --confidence-threshold {CONFIDENCE_THRESHOLD} \
    --log-level INFO

elapsed = time.time() - t_start
print(f"Pipeline finished in {elapsed:.1f} seconds ({elapsed/60:.2f} minutes)!")


## 7. Compute Official Penalized Accuracy (PA) & Benchmark Results
Evaluates predictions against ground truth answers using the competition metric:
$$\text{PA} = \frac{\text{Correct} - \text{Wrong}}{\text{Total}}$$

In [ ]:
import os
import json
import pandas as pd
from src.evaluate import compute_pa

SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.jsonl")
ANSWERS_PATH = os.path.join(DATA_DIR, "answers.jsonl")

if os.path.exists(SUBMISSION_PATH):
    submission_records = [json.loads(l) for l in open(SUBMISSION_PATH, encoding="utf-8-sig") if l.strip()]
    print(f"✅ Generated submission file: {SUBMISSION_PATH} with {len(submission_records)} records.")
    
    if os.path.exists(ANSWERS_PATH):
        preds = {r["id"]: r for r in submission_records}
        refs = {json.loads(l)["id"]: json.loads(l) for l in open(ANSWERS_PATH, encoding="utf-8-sig") if l.strip() and json.loads(l)["id"] in preds}
        metrics = compute_pa(preds, refs)
        total = metrics["total"]
        correct = metrics["correct"]
        wrong = metrics["wrong"]
        abstain = metrics["abstain"]
        pa = metrics["pa"]
        
        print("\n" + "=" * 60)
        print("🏆 DATAMATE OFFICIAL BENCHMARK SCORE (EVALUATED SUBSET)")
        print("=" * 60)
        print(f"Total Questions  : {total}")
        print(f"Correct (+1 pt)  : {correct} ({correct/total*100:.1f}%)")
        print(f"Wrong   (-1 pt)  : {wrong} ({wrong/total*100:.1f}%)")
        print(f"Abstained (0 pt) : {abstain} ({abstain/total*100:.1f}%)")
        print("-" * 60)
        print(f"🔥 Penalized Accuracy (PA): {pa:+.4f} ({pa*100:+.2f}%)")
        print("=" * 60)
        
        # Detailed breakdown by question type
        print("\n📊 DETAILED BREAKDOWN BY QUESTION TYPE:")
        type_rows = []
        for t, s in sorted(metrics.get("per_type", {}).items()):
            tot = s["total"]
            cor = s["correct"]
            wro = s["wrong"]
            abs_ = s["abstain"]
            sub_pa = (cor - wro) / tot if tot > 0 else 0.0
            type_rows.append({
                "Type": t,
                "Total": tot,
                "Correct": cor,
                "Wrong": wro,
                "Abstain": abs_,
                "PA": f"{sub_pa:+.4f}",
                "Acc%": f"{cor/tot*100:.1f}%" if tot > 0 else "0.0%"
            })
        df_types = pd.DataFrame(type_rows)
        print(df_types.to_string(index=False))
        print("=" * 60)
    else:
        print("\n" + "=" * 60)
        print("ℹ️ Mode: PUBLIC TEST (Ground truth answers are hidden on Zalo server)")
        print("=" * 60)
        types_count = {}
        for r in submission_records:
            t = r.get("type", "unknown")
            types_count[t] = types_count.get(t, 0) + 1
        print(f"Submission distribution across {len(submission_records)} questions:")
        for t, c in sorted(types_count.items(), key=lambda x: -x[1]):
            print(f"  - {t:15s}: {c:4d} ({c/len(submission_records)*100:5.1f}%)")
        print("=" * 60)
    
    # Save backup to Google Drive if available
    if os.path.exists("/content/drive/MyDrive/datamate"):
        !cp -f {SUBMISSION_PATH} /content/drive/MyDrive/datamate/submission_colab.jsonl
        print("💾 Saved submission copy to /content/drive/MyDrive/datamate/submission_colab.jsonl")
else:
    print(f"❌ Submission file not found at {SUBMISSION_PATH}")


## 8. Diagnostic Inspector: Error Analysis Breakdown

In [ ]:
import os
import json
import pandas as pd
from src.evaluate import score_answer

if os.path.exists(ANSWERS_PATH):
    # Load question text and business_id for context
    questions_map = {}
    if os.path.exists(QUESTIONS_FILE):
        with open(QUESTIONS_FILE, "r", encoding="utf-8-sig") as f:
            for line in f:
                if line.strip():
                    q = json.loads(line)
                    questions_map[q["id"]] = q

    preds = {r["id"]: r for r in submission_records}
    truths = {json.loads(l)["id"]: json.loads(l) for l in open(ANSWERS_PATH, encoding="utf-8-sig") if l.strip() and json.loads(l)["id"] in preds}

    wrong_list = []
    abstain_list = []

    for qid, p in preds.items():
        if qid not in truths:
            continue
        g = truths[qid]
        q_info = questions_map.get(qid, {})
        score = score_answer(p, g)
        
        p_val, g_val = p.get("answer"), g.get("answer")
        p_typ = p.get("type") or p.get("answer_type")
        g_typ = g.get("type") or g.get("answer_type")
        
        item = {
            "id": qid,
            "business_id": q_info.get("business_id", ""),
            "question": q_info.get("question", "")[:80],
            "pred_type": p_typ,
            "gt_type": g_typ,
            "pred_val": str(p_val)[:60],
            "gt_val": str(g_val)[:60],
        }
        
        if score == 0:
            abstain_list.append(item)
        elif score == -1:
            wrong_list.append(item)

    print(f"Total True Errors: {len(wrong_list)} | Total Abstains: {len(abstain_list)}")
    if wrong_list:
        df_err = pd.DataFrame(wrong_list)
        print("\n--- First 15 Incorrect Questions ---")
        display(df_err.head(15))
        
        err_path = os.path.join(OUTPUT_DIR, "error_analysis.csv")
        df_err.to_csv(err_path, index=False, encoding="utf-8-sig")
        print(f"📄 Saved error report to {err_path}")
        
    if abstain_list:
        df_abs = pd.DataFrame(abstain_list)
        abs_path = os.path.join(OUTPUT_DIR, "abstain_analysis.csv")
        df_abs.to_csv(abs_path, index=False, encoding="utf-8-sig")
        print(f"📄 Saved abstain report to {abs_path}")
else:
    print("ℹ️ Skipping error analysis (ground-truth answers not available for public_test).")


## 9. Backup All Results & Logs to Google Drive
Saves all prediction outputs, submission files, error analysis reports (`error_analysis.csv`), and execution logs (`predict.log`) to a timestamped folder and zip archive in Google Drive (`/content/drive/MyDrive/datamate/results/`).


In [ ]:
import os
import shutil
from datetime import datetime

DRIVE_BASE_DIR = "/content/drive/MyDrive/datamate"
RESULT_DIR = "/content/result"

if not os.path.exists("/content/drive/MyDrive"):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f"⚠️ Google Drive mount failed: {e}")

if os.path.exists(RESULT_DIR):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_drive_root = os.path.join(DRIVE_BASE_DIR, "results")
    timestamped_dir = os.path.join(results_drive_root, f"run_{timestamp}")
    latest_dir = os.path.join(results_drive_root, "latest")
    os.makedirs(timestamped_dir, exist_ok=True)
    os.makedirs(latest_dir, exist_ok=True)
    
    # 1. Copy all result files to timestamped directory & latest directory
    for item in os.listdir(RESULT_DIR):
        s = os.path.join(RESULT_DIR, item)
        d_time = os.path.join(timestamped_dir, item)
        d_late = os.path.join(latest_dir, item)
        if os.path.isdir(s):
            shutil.copytree(s, d_time, dirs_exist_ok=True)
            shutil.copytree(s, d_late, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d_time)
            shutil.copy2(s, d_late)
            
    # 2. Create a compressed zip backup
    zip_base = os.path.join(results_drive_root, f"results_{timestamp}")
    shutil.make_archive(zip_base, "zip", RESULT_DIR)
    
    print("=" * 60)
    print("✅ Successfully backed up all results to Google Drive!")
    print(f"📁 Timestamped Folder : {timestamped_dir}")
    print(f"📁 Latest Folder      : {latest_dir}")
    print(f"📦 Zip Archive        : {zip_base}.zip")
    print("=" * 60)
    !ls -lh {timestamped_dir}
else:
    print(f"❌ Result directory not found: {RESULT_DIR}")


## 10. 1-Click Submission File Download
Download the final `submission.jsonl` to your local machine for competition submission.


In [ ]:
from google.colab import files
import json

if os.path.exists(SUBMISSION_PATH):
    # Quick sanity check before download
    with open(SUBMISSION_PATH, "r", encoding="utf-8-sig") as f:
        lines = [l.strip() for l in f if l.strip()]
    print(f"Submission verification:")
    print(f"  - Total records: {len(lines)}")
    assert len(lines) > 0, "Submission file is empty!"
    
    # Check first record format
    first_record = json.loads(lines[0])
    assert "id" in first_record and "type" in first_record, f"Invalid format: {first_record}"
    print(f"  - Sample record: {first_record}")
    print("✅ All format checks passed! Ready for submission.")
    
    files.download(SUBMISSION_PATH)
    print(f"🚀 Downloading {SUBMISSION_PATH} to your browser...")
else:
    print(f"❌ File not found at {SUBMISSION_PATH}")
